In [ ]:
import os, shutil, sys
REPO_URL = "https://github.com/malaravanthulasi19-spec/oqmd-half-heusler-colab.git"
REPO_DIR = '/content/oqmd-half-heusler-colab'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone {REPO_URL} {REPO_DIR}
%cd /content/oqmd-half-heusler-colab
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
!pip install -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/oqmd_half_heusler_backups'
!mkdir -p {DRIVE_BACKUP_DIR}


In [ ]:
from pathlib import Path
from src.config import DEFAULT_DB_PATH, DEFAULT_CSV_PATH, DEFAULT_PARQUET_PATH
from src.database import OQMDDatabase
db = OQMDDatabase(DEFAULT_DB_PATH)
db.init_schema()
print(db.get_status())


In [ ]:
# Optional restore from Drive backup
BACKUP_DB = f'{DRIVE_BACKUP_DIR}/oqmd_half_heusler.sqlite3'
if Path(BACKUP_DB).exists():
    !cp {BACKUP_DB} {DEFAULT_DB_PATH}
    print('Restored DB from Drive backup')
else:
    print('No backup found yet')


In [ ]:
from src.downloader import OQMDDownloader
from src.oqmd_client import OQMDClient
client = OQMDClient(timeout_seconds=20, cooldown_on_502_seconds=60)
downloader = OQMDDownloader(db=db, client=client)
result = downloader.run()
print(result)
print('Safe to rerun this cell later if outage occurs.')


In [ ]:
from src.export_data import export_all
export_all(db, DEFAULT_CSV_PATH, DEFAULT_PARQUET_PATH)
!cp {DEFAULT_DB_PATH} {DRIVE_BACKUP_DIR}/oqmd_half_heusler.sqlite3
!cp {DEFAULT_CSV_PATH} {DRIVE_BACKUP_DIR}/oqmd_half_heusler.csv
!cp {DEFAULT_PARQUET_PATH} {DRIVE_BACKUP_DIR}/oqmd_half_heusler.parquet
print('Backup/export copied to Google Drive')
